### Feature Types
1. Raw features
    - age, salary, city

2. Aggregated features
    - total_spend
    - avg_salary_per_city

3. Temporal features
    - last_30d_spend
    - days_since_last_event

4. Ratio / interaction features
    - spend_per_order
    - salary_to_city_avg

5. Flags
    - is_high_earner
    - is_active_user

### Aggregation Features
#### Example: Orders per user
```python
user_features = (
    orders
    .groupby("user_id")
    .agg(
        total_amount=("amount", "sum"),
        order_count=("order_id", "count"),
        avg_amount=("amount", "mean")
    )
    .reset_index()
)
```

### Ratio & Interaction Features
```python
user_features["avg_order_value"] = (
    user_features["total_amount"] / user_features["order_count"]
)
```

Always guard against division by zero:
```python
user_features["avg_order_value"] = np.where(
    user_features["order_count"] > 0,
    user_features["total_amount"] / user_features["order_count"],
    0
)
```

### Time-Based Features
```python
orders["order_date"] = pd.to_datetime(orders["order_date"])
```

Leakage
```python
orders.groupby("user_id")["amount"].sum()
```
This uses future data.

**Cutoff-aware (Correct)**
```python
cutoff_date = "2024-01-01"

orders_before = orders[orders["order_date"] < cutoff_date]

features = (
    orders_before
    .groupby("user_id")["amount"]
    .sum()
)
```

### Rolling & Window Features
#### Example: Rolling 30-day spend
```python
orders = orders.sort_values(["user_id", "order_date"])

orders["rolling_30d_spend"] = (
    orders
    .groupby("user_id")["amount"]
    .rolling("30D", on="order_date")
    .sum()
    .reset_index(level=0, drop=True)
)
```

Requires:
- sorted data
- correct time index
- awareness of leakage

### Lag Features
#### Example: Previous order amount
```python
orders["prev_amount"] = (
    orders
    .groupby("user_id")["amount"]
    .shift(1)
)
```
Lag features:
- MUST create NaNs
- If no NaNs -> leakage occurred

### Frequency & Recency Features
```python
features = (
    orders
    .groupby("user_id")
    .agg(
        last_order_date=("order_date", "max"),
        order_count=("order_id", "count")
    )
)

features["days_since_last_order"] = (
    pd.Timestamp("2024-01-01") - features["last_order_date"]
).dt.days
```

### Categorical Encoding Features
```python
city_freq = df["city"].value_counts(normalize=True)
df["city_frequency"] = df["city"].map(city_freq)
```

### Base Datasets (Event + Entity)

#### Users (entity table - 1 row per user)

In [13]:
import pandas as pd
import numpy as np

users = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "signup_date": pd.to_datetime([
        "2023-01-01",
        "2023-01-10",
        "2023-01-20",
        "2023-02-01"
    ]),
    "city": ["Pune", "Mumbai", "Delhi", "Pune"]
})

#### Orders (event table - many rows per user)

In [14]:
orders = pd.DataFrame({
    "order_id": range(101, 111),
    "user_id": [1, 1, 2, 2, 2, 3, 3, 4, 4, 4],
    "order_date": pd.to_datetime([
        "2023-01-05", "2023-01-15",
        "2023-01-12", "2023-01-20", "2023-01-25",
        "2023-01-22", "2023-01-28",
        "2023-02-02", "2023-02-05", "2023-02-10"
    ]),
    "amount": [500, 700, 300, 400, 600, 1000, 1200, 200, 300, 400]
})

In [15]:
CUTOFF_DATE = pd.Timestamp("2023-02-01")

### Exercise 1
1. Filter orders to include **only events before** `CUTOFF_DATE`
1. Explain why `<= cutoff` is dangerous

In [16]:
orders_cutoff = orders[orders["order_date"] < CUTOFF_DATE]

Why `< cutoff`, not `<=`?
- `<=` may include events at **prediction time**
- In real systems, timestamps can be delayed or misaligned
- `< cutoff` is the safest leakage-free boundary

### Exercise 2
From filtered orders, compute per user:
- `total_spend`
- `order_count`
- `avg_order_value`

In [17]:
agg_features = (
    orders_cutoff.groupby("user_id")
    .agg(
        total_spend = ("amount", "sum"),
        order_count = ("order_id", "count"),
        avg_order_value = ("amount", "mean")
    )
    .reset_index()
)
agg_features

,user_id,total_spend,order_count,avg_order_value
0,1,1200,2,600.000000
1,2,1300,3,433.333333
2,3,2200,2,1100.000000


### Exercise 3
Create:
- `avg_order_value = total_spend / order_count`     
  
Rules:
- Guard against division by zero
- Explain why this guard matters

In [18]:
agg_features["avg_order_value"] = np.where(
    agg_features["order_count"] > 0,
    agg_features["total_spend"] / agg_features["order_count"],
    np.nan
)
agg_features

,user_id,total_spend,order_count,avg_order_value
0,1,1200,2,600.000000
1,2,1300,3,433.333333
2,3,2200,2,1100.000000


### Exercise 4
Create:
- `days_since_last_order`

Rules:
- Use `CUTOFF_DATE`
- Users with no orders -> `NaN`

Explain why recency is powerful.

In [19]:
last_order = (
    orders_cutoff.groupby("user_id")["order_date"]
    .max()
    .reset_index(name="last_order_date")
)

last_order["days_since_last_order"] = (
    CUTOFF_DATE - last_order["last_order_date"]
).dt.days

last_order

,user_id,last_order_date,days_since_last_order
0,1,2023-01-15,17
1,2,2023-01-25,7
2,3,2023-01-28,4


Why recency is powerful
- Strong proxy for **engagement**
- Used heavily in churn, fraud, recommendation systems

### Exercise 5
On the `orders` table:
1. Create `prev_order_amount`
2. Ensure:
    - NaNs appear
    - Data is sorted correctly

Explain:
- Why absence of NaNs indicates leakage

In [20]:
orders_sorted = orders.sort_values(["user_id", "order_date"])

orders_sorted["prev_order_amount"] = (
    orders_sorted
    .groupby("user_id")["amount"]
    .shift(1)
)

orders_sorted

,order_id,user_id,order_date,amount,prev_order_amount
0,101,1,2023-01-05,500,NaN
1,102,1,2023-01-15,700,500.0
2,103,2,2023-01-12,300,NaN
3,104,2,2023-01-20,400,300.0
4,105,2,2023-01-25,600,400.0
5,106,3,2023-01-22,1000,NaN
6,107,3,2023-01-28,1200,1000.0
7,108,4,2023-02-02,200,NaN
8,109,4,2023-02-05,300,200.0
9,110,4,2023-02-10,400,300.0


### Exercise 6
Create:
- `rolling_30d_spend` per user

Rules:
- Use rolling **time window**, not row window
- Respect cutoff
- Explain why rolling windows are dangerous

In [24]:
orders_sorted["rolling_30d_spend"] = (
    orders_sorted
    .groupby("user_id")
    .rolling(
        window="30D",
        on="order_date"
    )["amount"]
    .sum()
    .reset_index(level=0, drop=True)
)

Why rolling windows are dangerous
- Require strict sorting
- Easy to accidentally include future events
- Must always be cutoff-aware when used for ML

### Exercise 7
1. Merge all user-level features into users
1. Ensure:
   - One row per user
   - No duplicated users

Validate with:
```python
assert users_features["user_id"].is_unique
```

In [25]:
user_features = users.merge(
    agg_features,
    on="user_id",
    how="left"
).merge(
    last_order[["user_id", "days_since_last_order"]],
    on="user_id",
    how="left"
)

In [26]:
assert user_features["user_id"].is_unique

In [27]:
user_features

,user_id,signup_date,city,total_spend,order_count,avg_order_value,days_since_last_order
0,1,2023-01-01,Pune,1200.0,2.0,600.000000,17.0
1,2,2023-01-10,Mumbai,1300.0,3.0,433.333333,7.0
2,3,2023-01-20,Delhi,2200.0,2.0,1100.000000,4.0
3,4,2023-02-01,Pune,NaN,NaN,NaN,NaN
